
# 03 · 坐标帧与测量：同一个点到底在哪里？

上一单元的 `before_*` 是模拟器真实状态。现在先建立一个可检查的边界：车辆姿态在 world frame，控制器拿到的是传感器给出的 `DrivingObservation`。本课不训练视觉模型，而是显式生成有偏差、有噪声的测量，方便把误差和因果链一一对上。

每周约 26 小时的学习节奏中，本课建议 2 小时读公式与手算，3 小时改噪声参数并解释图，剩余时间写一页“测量误差如何改变动作”的实验笔记。


In [ ]:

from dataclasses import replace
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src" / "ad_tutorial").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from ad_tutorial.driving import DrivingObservation, DrivingConfig, run_episode
from ad_tutorial.estimation import (
    KalmanObserver, MeasurementConfig, RawMeasurementObserver, ScalarKalman,
    ego_to_world, make_observer, world_to_ego,
)



## 1. SE(2) 世界帧与自车帧

自车位置为 `t=(x,y)`、朝向为 `θ`。点 `p_w` 到 ego frame 的变换是

`p_e = R(θ)^T (p_w - t)`，其中 `R(θ)=[[cosθ,-sinθ],[sinθ,cosθ]]`。

`x_e` 指车头方向，`y_e` 指左侧。逆变换为 `p_w=R(θ)p_e+t`。先手算：自车在 `(10,2)`、朝向 90°，世界点 `(10,5)` 应在 ego `(3,0)`。


In [ ]:

ego_position = (10.0, 2.0)
heading = np.pi / 2
point_world = (10.0, 5.0)
point_ego = world_to_ego(point_world, ego_position, heading)
print("world -> ego:", point_ego)
assert np.allclose(point_ego, (3.0, 0.0), atol=1e-10)
assert np.allclose(ego_to_world(point_ego, ego_position, heading), point_world)



## 2. 测量模型：真值、偏差、随机噪声

我们写成 `z_k = x_k + b + v_k`：`b` 是固定偏差，`v_k ~ N(0,R)` 是每个采样独立的噪声。位置单位是 m，速度是 m/s，朝向是 rad；`MeasurementConfig` 保存各自带单位的 R/Q（Q 为每秒过程方差）、`seed` 和噪声参数。`lane_width_m` 是固定车道几何，因此不凭空加传感器误差。

同一 `seed` 的 raw 与 filter observer 每次 `reset()` 后会按同一顺序抽取噪声。两条闭环轨迹可能因动作不同而分叉，但第 k 行测量噪声仍有可比的 sample-index。进入闭环后，固定 lane 用 noisy world position 重新投影 `s/e_y`，因此不会把同一位置再独立加一份 lane 噪声。


In [ ]:

truth = DrivingObservation((10.0, 2.0), 0.1, 4.0, 10.0, 0.2, 0.0, 3.5)
sensor = MeasurementConfig(seed=23, position_bias_m=(0.0, 0.0),
                           speed_bias_mps=0.0)
raw = RawMeasurementObserver(sensor)
raw.reset(None)
samples = [raw.observe(truth, step, 0.1) for step in range(300)]
raw_rmse = np.sqrt(np.mean([(z.position[0] - truth.position[0]) ** 2 for z in samples]))
print(f"x-position raw RMSE = {raw_rmse:.3f} m; declared sigma = {sensor.position_noise_std_m[0]:.3f} m")



固定偏差不会被随机滤波自动标定。保持 stationary truth 和同一个 noise seed，只加入 `+0.20m` 的 world-x bias；比较 raw 与 filter 的 signed mean error。filter 可以压低抖动，最终仍会围绕带偏差的位置。


In [ ]:

biased = MeasurementConfig(seed=23, position_bias_m=(0.20, 0.0),
                           heading_bias_rad=0.0, speed_bias_mps=0.0)
raw_biased = RawMeasurementObserver(biased)
filter_biased = KalmanObserver(biased)
raw_biased.reset(None); filter_biased.reset(None)
raw_samples = [raw_biased.observe(truth, step, 0.1) for step in range(300)]
filter_samples = [filter_biased.observe(truth, step, 0.1) for step in range(300)]
raw_signed = np.mean([z.position[0] - truth.position[0] for z in raw_samples[20:]])
filter_signed = np.mean([z.position[0] - truth.position[0] for z in filter_samples[20:]])
print(f"signed x bias: raw={raw_signed:.3f}m, filter={filter_signed:.3f}m")
assert abs(raw_signed - 0.20) < 0.05 and abs(filter_signed - 0.20) < 0.05



## 3. 可编辑练习：先预测再运行

1. 把 `heading` 改为 0，世界点 `(13,2)` 的 ego 坐标是什么？
2. 只把 `position_noise_std_m` 的 x 分量从 0.18 改成 0.40。raw position RMSE 会如何变化？偏差是否会被更多样本平均掉？
3. 用 `world_to_ego` 与 `ego_to_world` 做 100 个随机点的逆变换测试。

答案：第 1 题是 `(3,0)`；第 2 题 RMSE 通常随噪声标准差增大，固定 bias 不会因取平均自动消失；第 3 题误差应在浮点精度内。不要把这些答案当作真实传感器标定结论。


In [ ]:

rng = np.random.default_rng(4)
round_trip_error = []
for _ in range(100):
    pose = rng.normal(size=2)
    angle = rng.uniform(-np.pi, np.pi)
    point = rng.normal(size=2)
    round_trip_error.append(np.linalg.norm(ego_to_world(world_to_ego(point, pose, angle), pose, angle) - point))
print("max inverse error:", max(round_trip_error))



朝向是圆变量。实现会把新测量相对上一次估计的 innovation wrap 到 `[-π,π)`；因此 `+π−0.02` 到 `−π+0.02` 是约 0.04 rad 的小变化。CLI 的 heading RMSE 也使用同样的 wrapped error。


In [ ]:

boundary = MeasurementConfig(seed=2, position_noise_std_m=(0.0, 0.0),
                             heading_noise_std_rad=0.05, speed_noise_std_mps=0.0,
                             position_bias_m=(0.0, 0.0), heading_process_variance_rad2_per_s=0.0001)
crossing = KalmanObserver(boundary)
crossing.reset(None)
crossing.observe(replace(truth, heading_rad=np.pi - 0.02), 0, 0.1)
estimate = crossing.observe(replace(truth, heading_rad=-np.pi + 0.02), 1, 0.1).heading_rad
wrapped_error = (estimate - (-np.pi + 0.02) + np.pi) % (2*np.pi) - np.pi
print("wrapped heading error:", wrapped_error)
assert abs(wrapped_error) < 0.15



## 4. 研究入口

TUMFTM 的 [Autonomous Driving Software Engineering](https://github.com/TUMFTM/Lecture_ADSE) 在 mapping/localization 章节给出 GNSS 与 Kalman filter 的工程练习；Simo Särkkä 与 Lennart Svensson 的 [Bayesian Filtering and Smoothing](https://users.aalto.fi/~ssarkka/pub/bfs_book_2023_online.pdf) 是系统推导。阅读时只需回答：状态转移模型、测量模型、协方差各自描述什么？本课的标量随机游走模型做了哪些简化？
